In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('../../music-sources-unified'))
if module_path not in sys.path: sys.path.append(module_path)
import unify_lib as uni

import pandas as pd
import time


ytmusic_path = os.path.join('../../ytmusic/')
yt_module_path = os.path.abspath(ytmusic_path)
if yt_module_path not in sys.path:  sys.path.append(yt_module_path)
from ytmusic_library import YTMusicPlaylists, PLAYLIST_TSV_COLUMNS


HEADER_FILE=os.path.join(ytmusic_path, 'headers_auth.json')
BACKUP_DIR = os.path.join(ytmusic_path, 'playlists/')
PLAYCOUNT_FILE=os.path.join(BACKUP_DIR, '_ytmusic_lastfm_match_id_map.tsv')
NOT_LIKE_PLAYLIST_TSV =os.path.join(BACKUP_DIR, 'zz not like.tsv')
LIKE_PLAYLIST_TSV = os.path.join(BACKUP_DIR, '_liked_tracks.tsv')
LIKE_YT_PLAYLIST_TSV = os.path.join(BACKUP_DIR, 'Liked Music.tsv')
ALL_TRACKS_TSV = os.path.join(BACKUP_DIR, '_tracks_db.tsv')
PLAYLIST_RADIO_COUNT_TSV=os.path.join(BACKUP_DIR, '_playlist_radio_counts.tsv')
RADIO_TO_LIKE_PL_TSV=os.path.join(BACKUP_DIR, '_ytmusic_radio_to_like_pl_map.tsv')

MANUALLY_RATED_TSV = '../tsvs/ytmusic_new_like_and_not_like_manual_rated.tsv'
NEED_RATE_TSV =  '../tsvs/ytmusic_new_like_and_not_like_need_manual_rating.tsv'

Y = YTMusicPlaylists(header=HEADER_FILE, playcount_map=PLAYCOUNT_FILE,  not_like_tsv=NOT_LIKE_PLAYLIST_TSV)
print(f"Loaded {len(Y.playlists['title'].unique())} playlists")

In [8]:
RADIO_TO_LIKE_PL_TSV='../../ytmusic/playlists/_ytmusic_radio_to_like_pl_map.tsv'
map = pd.read_csv(RADIO_TO_LIKE_PL_TSV, sep='\t')
map

,radio_playlist,like_playlist
0,x_r.minimal_tracks_radio,x_r.minimal_tracks_like
1,x_r.deepcuts_tracks_radio,x_r.deepcuts_tracks_like
2,Bossa Nova radio,Bossa Nova
3,Hip Hop Hits radio,Hip Hop Hits
4,rock stoner sludge dank radio,rock stoner sludge dank
...,...,...
176,x_r.dub_tracks_radio,Reggae Dub
177,triphop bristol sound radio,triphop bristol sound
178,jazz essential radio,jazz
179,Hiphop southeast Ride Around Shining radio,hiphop 2000s southern


## Like not liker miner

Get all ytmusic entries, get like entries and aget not like entries, create artist - track key and move all matching likes and not like to respective playlist

In [150]:

# like_df = pd.read_csv(LIKE_PLAYLIST_TSV, sep='\t', index_col=0)
# Concat like playlists together and remove dupes
yt_like_df = pd.read_csv(LIKE_YT_PLAYLIST_TSV, sep='\t', index_col=0)
yt_like_df = yt_like_df.set_index('videoId', drop=True)
like_df =  pd.read_csv(LIKE_PLAYLIST_TSV, sep='\t', index_col=0)
like_df = pd.concat([like_df, yt_like_df[like_df.columns]])
like_df = like_df[~like_df.index.duplicated(keep='first')]
del yt_like_df

not_like_df = pd.read_csv(NOT_LIKE_PLAYLIST_TSV, sep='\t', index_col=0)
not_like_df = not_like_df.set_index('videoId', drop=True)
all_df = pd.read_csv(ALL_TRACKS_TSV, sep='\t', index_col=0)

print(f'Loaded {len(like_df)} like, {len(not_like_df)} not like entries, and {len(all_df)} total tracks')
not_like_df = not_like_df.loc[not_like_df['likeStatus'] !='LIKE']
print(f'Keeping {len(not_like_df)} not like after remove LIKE')
# Add id
like_df['fuzzy_track_id'] = like_df.apply(uni.make_ytmusic_fuzzy_slugified_track_id, axis=1)
not_like_df['fuzzy_track_id'] = not_like_df.apply(uni.make_ytmusic_fuzzy_slugified_track_id, axis=1)
all_df['fuzzy_track_id'] = all_df.apply(uni.make_ytmusic_fuzzy_slugified_track_id, axis=1)

# Sets for quick lookup
not_like_vids = frozenset(not_like_df.index)
not_like_fuzzy_ids = frozenset(not_like_df['fuzzy_track_id'])
like_vids = frozenset(like_df.index)
like_fuzzy_ids = frozenset(like_df['fuzzy_track_id'])

# v0
# Loaded 39646 like, 4531 not like entries, and 149039 total tracks
# Keeping 4383 not like after remove LIKE


Loaded 41461 like, 4806 not like entries, and 149082 total tracks
Keeping 4613 not like after remove LIKE


In [151]:
print('Looking through LIKE tracks (~7m)')
like_impacted_playlists = []
new_likes = set()
skip_not_like = set()
for fuzzy_track_id in like_fuzzy_ids:
  matches = all_df.loc[all_df['fuzzy_track_id'] == fuzzy_track_id]
  if not len(matches): continue
  for match in matches.itertuples():
    if match.likeStatus == 'LIKE': continue
    if match.Index in like_vids: continue
    match_name = f'{match.artist} - {match.album} - {match.title}'
    if match.Index in not_like_vids:
      skip_not_like.add(match.Index)
      continue
    new_likes.add(match.Index)
    if pd.isna(match.playlists): continue
    like_impacted_playlists.append(match.playlists) 
print(f' Found {len(new_likes)} new tracks to LIKE')
print(f' Found {len(skip_not_like)} new tracks to LIKE but they are already in NOT LIKE')

print(100*'*')
print('Looking through NOT LIKE tracks (~1m)')
not_like_impacted_playlists = []
new_not_likes = set()
skip_is_like = set()
for fuzzy_track_id in not_like_fuzzy_ids:
  matches = all_df.loc[all_df['fuzzy_track_id'] == fuzzy_track_id]
  if not len(matches): continue
  for match in matches.itertuples():
    if match.Index in not_like_vids: continue
    match_name = f'{match.artist} - {match.album} - {match.title}'
    if match.Index in like_vids or match.Index in new_likes :
      skip_is_like.add(match.Index)
      continue
    new_not_likes.add(match.Index)
    if not pd.isna(match.playlists): continue
    not_like_impacted_playlists.append(match.playlists) 


print(f' Found {len(new_not_likes)} new tracks to NOT LIKE')
print(f' Found {len(skip_is_like)} new tracks to NOT LIKE but they are already in LIKE')

def generate_impacted_playlist_df(impacted_playlists):
  df = pd.DataFrame(impacted_playlists, columns=['encoded_list'])
  df['encoded_list'] = df['encoded_list'].dropna().str.replace('[nan]', "['nan']")
  df['decoded_list'] = df['encoded_list'].str.lstrip('[').str.rstrip(']').str.split(', ')
  df = df.explode('decoded_list')
  return df['decoded_list'].str.strip("'")

print(100*'*')
like_impacted_playlists_df = generate_impacted_playlist_df(like_impacted_playlists)
print(f'Top 10 playlists impacted by LIKE: {like_impacted_playlists_df.value_counts().head(10)}')


Looking through LIKE tracks (~5m)
 Found 376 new tracks to LIKE
 Found 166 new tracks to LIKE but they are already in NOT LIKE
****************************************************************************************************
Looking through NOT LIKE tracks (~1m)
 Found 120 new tracks to NOT LIKE
 Found 228 new tracks to NOT LIKE but they are already in LIKE
****************************************************************************************************
Top 20 playlists impacted by LIKE: decoded_list
nan                                   37
y_2012_thumbs_up                      20
y_2014_thumbs_up                      17
beats                                 16
psych rock modern                     12
psychedelic rock mega                 11
zzzz_all 6                            11
y_2017_thumbs_up                      11
y_2013_thumbs_up                      10
zz__thumbs_up                         10
rock modern chill                     10
indie loose like                     

In [158]:
# Load previoius manual ratings and remove ones already in v0 batch
# use this to remove tracks already rated, then add ones that need rating to end or something
manual_picks = pd.read_csv(MANUALLY_RATED_TSV, sep='\t', index_col=0).drop_duplicates(keep='first') # remove duplicate index
old_like = manual_picks.loc[manual_picks['manual_rating'] == 'LIKE']
old_not_like = manual_picks.loc[manual_picks['manual_rating'] == 'NOT_LIKE']
# Loaded 7734 manually labeled entries, 2839 are LIKE, 4850 NOT_LIKE
print(f'Loaded {len(manual_picks)} already  manually labeled entries, {len(old_like)} are LIKE, {len(old_not_like)} NOT_LIKE')

old_like_vids = frozenset(old_like.index)
old_not_like_vids = frozenset(old_not_like.index)
old_all = old_like_vids & old_not_like_vids

new_likes_len = len(new_likes)
new_likes = new_likes- old_like_vids
print(f'Reduced new LIKE from {new_likes_len} to {len(new_likes)} entries after removing already reviewed matches')


new_not_likes_len = len(new_not_likes)
new_not_likes = new_not_likes- old_not_like_vids
print(f'Reduced new NOT_LIKE from {new_not_likes_len} to {len(new_not_likes)} entries after removing already reviewed matches')


skip_not_like_len = len(skip_not_like)
skip_not_like = skip_not_like- old_all
print(f'Reduced skip_not_like from {skip_not_like_len} to {len(skip_not_like)} entries after removing already reviewed matches')

skip_is_like_len = len(skip_is_like)
skip_is_like = skip_is_like- old_all
print(f'Reduced skip_like from {skip_is_like_len} to {len(skip_is_like)} entries after removing already reviewed matches')

# Loaded 7734 already  manually labeled entries, 2839 are LIKE, 4850 NOT_LIKE
# Reduced new LIKE from 376 to 56 entries after removing already reviewed matches
# Reduced new NOT_LIKE from 120 to 10 entries after removing already reviewed matches
# Reduced skip_not_like from 158 to 158 entries after removing already reviewed matches
# Reduced skip_like from 228 to 207 entries after removing already reviewed matches

Loaded 7734 already  manually labeled entries, 2839 are LIKE, 4850 NOT_LIKE
Reduced new LIKE from 376 to 56 entries after removing already reviewed matches
Reduced new NOT_LIKE from 120 to 10 entries after removing already reviewed matches


In [ ]:
# Manually screeen these in sheets, llabel as LIKE, INDIFFERENT or NOT_LIKE 
# # Sets
# need_like: Most of these were LIKE, but some overridden to NOT_LIKE or INDIFFERENT (around 2000)
# need_not_like: Most of these were NOT_LIKE, but some overridden t0 INDIFFERENT (around 4000)
# need_like_but_is_not_like: Manualy reviewed (around 100)
# need_not_like_but_is_like: Manualy reviewed (around 100)
like_not_like_res = {
 
  'need_like': all_df.loc[all_df.index.isin(new_likes)],
  'need_like_but_is_not_like': all_df.loc[all_df.index.isin(skip_not_like)],
  'need_not_like': all_df.loc[all_df.index.isin(new_not_likes)],
  'need_not_like_but_is_like': all_df.loc[all_df.index.isin(skip_is_like)],
}


save_cols = ['title',  'artist', 'album', 'albumArtist', 
             'likeStatus',  'averageRating',
             'albumYear', 'albumType','duration_seconds']

need_review = []
for k, v in like_not_like_res.items():
  v['set'] = k
  need_review.append(v)
pd.concat(need_review).to_csv(NEED_RATE_TSV, sep='\t', index=True)
  

# Saving _need_like tsv with 2868 entries
# Saving _need_like_but_is_not_like tsv with 187 entries
# Saving _need_not_like tsv with 4749 entries
# Saving _need_not_like_but_is_like tsv with 259 entries


In [ ]:
# use this to remove tracks already rated, then add ones that need rating to end or something
manual_picks = pd.read_csv(MANUALLY_RATED_TSV, sep='\t', index_col=0).drop_duplicates(keep='first') # remove duplicate index
to_like = manual_picks.loc[manual_picks['manual_rating'] == 'LIKE']
to_not_like = manual_picks.loc[manual_picks['manual_rating'] == 'NOT_LIKE']
# Loaded 7734 manually labeled entries, 2839 are LIKE, 4850 NOT_LIKE
print(f'Loaded {len(manual_picks)} manually labeled entries, {len(to_like)} are LIKE, {len(to_not_like)} NOT_LIKE')


to_like_pl_id = Y.yt.create_playlist(
  title='_likes_new', video_ids=list(to_like.index), privacy_status='PRIVATE',
  description=f'{len(to_like)} tracks that should be like')
to_not_like_pl_id = Y.yt.create_playlist(
  title='_not_likes_new', video_ids=list(to_not_like.index), privacy_status='PRIVATE',
  description=f'{len(to_not_like)} tracks that should be not like')


# Like the to_like playlist
# Playlist _likes_new: Rated 2708 of 2839 tracks as LIKE
Y.playlist_rate_all_songs(Y.playlist_get_info(to_like_pl_id), 'LIKE', sleep_time=0.5,  verbose=False, skip_if_dislike=False)

# delete to like
# merge not like with xx? or split into 2 art?


## Fix existing playlists

In [170]:
not_like_df = pd.read_csv(NOT_LIKE_PLAYLIST_TSV, sep='\t', index_col=0)
not_like_vids = frozenset(not_like_df.videoId)
print(f'Loaded {len(not_like_vids)} not like entries')
radio_to_like_df = pd.read_csv(RADIO_TO_LIKE_PL_TSV, sep='\t')
no_matchlike_pl = radio_to_like_df.loc[radio_to_like_df['like_playlist'].isna()]
print(f'Loaded {len(radio_to_like_df)} radio to like playlist matches, {len(no_matchlike_pl)} do not have a match')
completed = {}
# Loaded 4806 not like entries
# Loaded 181 radio to like playlist matches, 28 do not have a match

Loaded 4806 not like entries
Loaded 181 radio to like playlist matches, 28 do not have a match


### TODO integrate new functions in Y which were copied from here, also move this notbook back to ytmusisc (and maybe some assets)

In [ ]:
VERBOSE = True
REMOVE_NOT_LIKE_AND_DISLIKE = True
MOVE_LIKE = True
MIN_NUM_LIKE = 10
CREATE_LIKE_PLAYLIST = True

counter_df = []
like_not_like_vids = []
for pl in Y.playlists.itertuples():
  if 'radio' not in pl.title: continue

  if pl.title in completed: 
    if VERBOSE: print(f'Already completed {pl.title}')
    continue

  like_pl = radio_to_like_df.loc[radio_to_like_df['radio_playlist'] == pl.title].iloc[0]['like_playlist']

  pl_info = Y.playlist_get_info(pl.playlistId, use_cache=True)
  remove_tracks = []
  move_like_tracks = []
  pl_counters = {'name': pl.title, 'removed_dislike': 0, 'moved_like': 0, 'removed_not_like': 0, 'like_and_not_like': 0}
  for track in pl_info.get('tracks', []):
    if track['likeStatus'] == 'DISLIKE':
      pl_counters['removed_dislike'] += 1
      remove_tracks.append(track)
    elif track['likeStatus'] == 'LIKE':
      pl_counters['moved_like'] += 1
      move_like_tracks.append(track)
      if track['videoId'] in not_like_vids:
        pl_counters['like_and_not_like'] += 1
        like_not_like_vids.append(track['videoId'])
    elif track['videoId'] in not_like_vids:
      pl_counters['removed_not_like'] += 1
      remove_tracks.append(track)     
  counter_df.append(pl_counters)
  if VERBOSE: print(100*'*' + f'\n{pl_counters}')
  
  # Handle flagged tracks
  if MOVE_LIKE and len(move_like_tracks) > MIN_NUM_LIKE:
    # Create like playlist and add from 'move_like_tracks'
    like_pl_id = None
    like_vids = [t['videoId'] for t in move_like_tracks]
    if pd.isna(like_pl):
      if CREATE_LIKE_PLAYLIST:
        like_pl = pl.title.replace('radio', 'like')
        like_pl_id = Y.yt.create_playlist(
            title=like_pl,  description=f'Created for dumping likes from {pl.title}',
            privacy_status='PRIVATE', video_ids=like_vids)
        if VERBOSE: print(f'Created LIKE playlist for {pl.title}: {like_pl}')
        time.sleep(3)

    else:
      like_pl_id = Y.query_by_title(like_pl).playlistId
      like_orig_vids = frozenset([t['videoId'] for t in Y.playlist_get_info(like_pl_id, use_cache=False).get('tracks', [])])
      like_new_vids = frozenset(like_vids) - like_orig_vids
      like_dedupe_num = len(like_vids) - len(like_new_vids)
      if like_dedupe_num > 0:
        like_vids = list(like_new_vids)
        
      if len(like_vids):
        status = Y.yt.add_playlist_items(playlistId=like_pl_id, videoIds=like_vids, duplicates=False)
        # Somtimes this still fails, fallback is to reemove like pl mapping so it generates a fresh pl
        assert status['status'] == 'STATUS_SUCCEEDED', f'Bad Status for {pl.title} add {len(move_like_tracks)} LIKE tracks: {status}'
        time.sleep(1)
      elif VERBOSE: 
        print(f'No new LIKE tracks to add to playlist {like_pl_id}')
        
    if like_pl_id == None:
      if VERBOSE: print(f'No LIKE playlist for {pl.title}, so not moving {len(move_like_tracks)} LIKE tracks')
      continue
    if VERBOSE: print(f'Added {len(move_like_tracks)} LIKE entries from {pl.title} to {like_pl_id}')
    
    status = Y.yt.remove_playlist_items(pl.playlistId, move_like_tracks)
    assert str(status) == 'STATUS_SUCCEEDED', f'Bad Status for {pl.playlistId} remove {len(move_like_tracks)} LIKE tracks: {status}'
    time.sleep(1)
    if VERBOSE: print(f'Moved {len(move_like_tracks)} LIKE entries from {pl.title}')
    
  if REMOVE_NOT_LIKE_AND_DISLIKE and len(remove_tracks):
    status = Y.yt.remove_playlist_items(pl.playlistId, remove_tracks)
    assert str(status) == 'STATUS_SUCCEEDED', f'Bad Status for {pl.playlistId} remove {len(remove_tracks)} NOT LIKE tracks: {status}'
    if VERBOSE: print(f'Removed {len(remove_tracks)} NOT_LIKE entries from {pl.title}')    
    time.sleep(1)
  completed[pl.title] = pl_counters
  

complete_df = pd.DataFrame(completed).T[['removed_dislike', 'moved_like', 'removed_not_like', 'like_and_not_like', 'status']].sort_values('moved_like')
complete_df['total_changes'] = complete_df[['removed_dislike', 'moved_like', 'removed_not_like', 'like_and_not_like']].sum(axis=1)
complete_df = complete_df.sort_values('total_changes', ascending = False)
complete_df.to_csv('../logs/ytmusic_match_like_not_like_result.tsv', sep='\t')


### Count number of tracks in radio playlists
(takes 5m)

In [174]:
playlists = Y.get_playlist_counts(verbose=False, filter_title='radio')
radio_counts_df = playlists.loc[playlists.title.str.contains('radio')].sort_values('track_count')
radio_counts_df = radio_counts_df[['title', 'track_count', 'duration_hours', 'privacy', 'playlist_id']]
radio_counts_df.to_csv(PLAYLIST_RADIO_COUNT_TSV, sep='\t', index=False)
radio_counts_df
# last run 7-2023

,title,playlist_id,track_count,privacy,duration_hours
67,Reggae 1970 roots radio,PLWptjpDqazOxNFmvqxnS9RYTsm51yErv6,0,PRIVATE,0
0,ambient Indie Synths radio,PLWptjpDqazOw7al5TJVXbaiQr1ILDBk_y,27,PRIVATE,2
44,Jazz Feels the Blues radio,PLWptjpDqazOwjAXRmi4n6wLMiV9hsm1pT,27,PRIVATE,3
82,Soul Food Kitchen radio,PLWptjpDqazOxDwaPfQXeeho7O2tLmgjP_,28,PRIVATE,2
71,rock 1967 Monterey Pop Festival radio,PLWptjpDqazOyoXjkoYrtOq-HxIFikPMb0,33,PRIVATE,2
...,...,...,...,...,...
87,x_r.2010smusic_tracks_radio,PLWptjpDqazOyU9d_yupbcHOq-9MYYOonW,1106,PRIVATE,72
122,x_r.futuregarage_tracks_radio,PLWptjpDqazOxj9ct9vG8tMeOMdj2rH_PQ,1142,PRIVATE,97
150,x_r.reggae_tracks_radio,PLWptjpDqazOxFpTJUsBAOp163DArs5Wyf,1198,PRIVATE,83
69,Reggae radio,PLWptjpDqazOwE761BnO1IfHJxwd9W8waZ,1351,PRIVATE,93
